# Neural Network Model of NBA All-Star Predictions: 

## Libraries/Requirements: 

In [52]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score,
    f1_score, accuracy_score
)
import os
from pathlib import Path
import torch
import torch.nn as nn

## Deterministic Setup and Hyperparameter Selection: 

In [53]:
class Config:
    """
        Central configuration for the neural network model.
        Defines training hyperparameters, model architecture, loss weighting,
        data split boundaries, and runtime device. The goal is to keep all
        experimental knobs in one place for reproducibility and clarity.
        Notes:
        - HIDDEN_DIMS controls the MLP depth/width before attention.
        - ATTN controls interaction modeling across players in a group.
        - TOPK_TEMP controls sharpness of soft selection.
        - Loss weights balance ranking vs calibration vs structured selection.
        - TRAIN_END / VAL_END enforce temporal splits to avoid leakage.
    """
    SEED = 47
    LR = 0.00005
    WEIGHT_DECAY = 0.00001
    EPOCHS = 80
    HIDDEN_DIMS = [256, 128, 64]
    DROPOUT = 0.1
    ATTN_HEADS = 4
    ATTN_DIM = ATTN_HEADS * 32
    TOPK_TEMP = 0.34
    PAIRWISE_WEIGHT = 0.3
    BCE_WEIGHT = 0.4
    GRAD_CLIP = 1.0
    TRAIN_END = 2015
    VAL_END = 2021
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    DATA_PATH = 'source\\cleaned\\cleaned_data.csv'

def set_seed(seed):
    """
        Fixes all major sources of randomness for reproducibility.
        Ensures consistent results across:
        - PyTorch (CPU + CUDA)
        - NumPy
        - cuDNN backend
        This makes training deterministic at the cost of some performance.
    """
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def build_group_indices(df, device):
    """
        Groups indices by (season, conference).
        Returns a list of index tensors, where each tensor corresponds to a
        single selection pool (i.e., one conference in one season).
        This is critical because:
        - Attention is applied within groups
        - Loss functions (top-k, pairwise) operate at the group level
        - Selection constraints are enforced per group
        Output:
            List[Tensor]: each tensor contains indices for one group
    """
    keys = df[['Season Ending Year', 'Conference_East']].values

    groups = {}
    for i, k in enumerate(map(tuple, keys)):
        groups.setdefault(k, []).append(i)

    return [
        torch.tensor(v, dtype=torch.long, device=device)
        for v in groups.values()
    ]

def get_project_root():
    """
        Dynamically locates the project root directory.
        Walks up the directory tree until a folder containing 'source/'
        is found. This avoids hardcoding file paths and makes the project
        portable across environments (local, notebook, different machines).
        Raises:
            FileNotFoundError if the expected structure is not found.
    """
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / "source").exists():  
            return parent
    raise FileNotFoundError("Could not find project root (missing 'source' folder)")

ROOT = get_project_root()
DATA_PATH = ROOT / "source" / "cleaned" / "cleaned_data.csv"

The Configuration class defined above enables modular setups and experiments, and reproducible and consistent results across different runs. This then allows for future work and for exploration of the importance of parameter values, which are defined as follows: 

| Component        | Parameter        | Value            | Meaning |
|------------------|------------------|------------------|--------|
| Reproducibility  | Seed             | 47               | Fixes randomness across NumPy and PyTorch |
| Training         | Learning Rate    | 5e-5             | Keeps updates small to stabilize training |
| Training         | Weight Decay     | 1e-5             | L2 regularization to control overfitting |
| Training         | Epochs           | 80               | Enough to converge without drifting |
| Architecture     | Hidden Dims      | [256, 128, 64]   | Expands then compresses representation |
| Architecture     | Dropout          | 0.1              | Light regularization without underfitting |
| Attention        | Heads            | 4                | Captures multiple interaction patterns |
| Attention        | Dimension        | 128              | Total embedding size across heads |
| Objective        | Temperature      | 0.34             | Sharpens soft top-k selection |
| Objective        | Pairwise Weight  | 0.3              | Pushes true All-Stars above others |
| Objective        | BCE Weight       | 0.4              | Keeps probabilities calibrated |
| Optimization     | Grad Clip        | 1.0              | Prevents unstable gradients |
| Data Split       | Train End        | 2015             | Trains only on past seasons |
| Data Split       | Val End          | 2021             | Holds out recent seasons for tuning |
| Hardware         | Device           | GPU / CPU        | Uses GPU if available |
| Structure        | Grouping         | (Season, Conf)   | Defines each selection pool |
| Data             | Path             | relative to root | Makes the pipeline portable |

## Data Pipeline: 

In [54]:
class DataPipeline:
    """
        End-to-end data processing pipeline.
        Handles:
        - feature construction (domain + contextual features)
        - temporal splitting (train / val / test)
        - normalization and tensor conversion
        The design emphasizes relative performance within a season/conference
        and preserves group structure for downstream attention and selection.
    """
    def __init__(self, cfg):
        """
            Initializes preprocessing components.
            Uses standard scaling to normalize feature magnitudes,
            which stabilizes optimization for the neural network.
        """
        self.cfg = cfg
        self.scaler = StandardScaler()

    def load(self, path):
        """
            Loads raw CSV data and applies feature engineering.
            Args:
                path (str or Path): location of cleaned dataset
            Returns:
                DataFrame with engineered features
        """
        df = pd.read_csv(path)
        return self._features(df)

    def _features(self, df):
        """
            Constructs domain-informed and contextual features.
            Key ideas:
            - Encode categorical structure (conference, position, season)
            - Capture usage and scoring impact
            - Normalize performance within (season, conference)
            - Add temporal signals (previous All-Star selections)
            Feature groups:
            - IDs: Conf_ID, Pos_ID, Season_ID (for embeddings)
            - Usage/impact: Usage, Usage_x_Win, Impact_on_winning
            - Relative scoring: PTS_conf_z, PTS_conf_share
            - Relative impact: Impact_conf_z
            - Temporal: AllStar_Last_Year, AllStar_last_2
            Returns:
                DataFrame with additional feature columns
        """

        df['Conf_ID'] = df['Conference_East'].astype(int)
        df['Pos_ID'] = df['PosGroup_Backcourt'].astype(int)

        min_year = df['Season Ending Year'].min()
        df['Season_ID'] = df['Season Ending Year'] - min_year

        df['Usage'] = df['FGA per game'] + 0.44 * df['FTA per game']
        df['Usage_x_Win'] = df['Usage'] * df['Team Win %']
        df['Impact_on_winning'] = df['PTS per game'] * df['Team Win %']

        group_keys = ['Season Ending Year', 'Conference_East']

        pts_mean = df.groupby(group_keys)['PTS per game'].transform('mean')
        pts_std = df.groupby(group_keys)['PTS per game'].transform('std')

        df['PTS_conf_z'] = (df['PTS per game'] - pts_mean) / (pts_std + 1e-8)

        pts_max = df.groupby(group_keys)['PTS per game'].transform('max')
        df['PTS_conf_share'] = df['PTS per game'] / (pts_max + 1e-8)

        impact_mean = df.groupby(group_keys)['Impact_on_winning'].transform('mean')
        impact_std = df.groupby(group_keys)['Impact_on_winning'].transform('std')

        df['Impact_conf_z'] = (
            (df['Impact_on_winning'] - impact_mean) /
            (impact_std + 1e-8)
        )

        df = df.sort_values(['Player', 'Season Ending Year']).reset_index(drop=True)

        df['AllStar_Last_Year'] = (
            df.groupby('Player')['All Star'].shift(1).fillna(0)
        )

        df['AllStar_last_2'] = (
            df.groupby('Player')['All Star']
            .shift(1)
            .rolling(2)
            .sum()
            .fillna(0)
        )

        return df

    def split(self, df):
        """
            Splits data temporally into train, validation, and test sets.
            This prevents leakage across seasons and ensures that
            evaluation reflects forward prediction.
            Splits:
            - Train: ≤ TRAIN_END
            - Validation: (TRAIN_END, VAL_END]
            - Test: > VAL_END
        """
        train_df = df[df['Season Ending Year'] <= self.cfg.TRAIN_END].reset_index(drop=True)
        val_df = df[
            (df['Season Ending Year'] > self.cfg.TRAIN_END) &
            (df['Season Ending Year'] <= self.cfg.VAL_END)
        ].reset_index(drop=True)
        test_df = df[df['Season Ending Year'] > self.cfg.VAL_END].reset_index(drop=True)

        return train_df, val_df, test_df

    def prepare(self, train_df, val_df, test_df):
        """
            Prepares data for model input.
            Steps:
            1. Separate features, labels, and embedding indices
            2. Impute missing values using training medians
            3. Normalize features using training statistics
            4. Convert all arrays to PyTorch tensors on the target device
            Outputs:
            - X: normalized feature tensors
            - y: labels
            - c, p, s: categorical indices (conference, position, season)
            Notes:
            - Categorical IDs are kept separate for embedding layers
            - Scaling is fit on train only to avoid leakage
            - All tensors are moved to DEVICE for efficient training
        """
        
        drop_cols = [
            'All Star', 'Player', 'Season Ending Year',
            'Prev All Stars', 'Conference_East',
            'PosGroup_Backcourt', 'PosGroup_Frontcourt'
        ]
       
        def split_xy(df):
            X = df.drop(columns=drop_cols + ['Conf_ID', 'Pos_ID', 'Season_ID'])
            y = df['All Star']
            return X, y, df['Conf_ID'], df['Pos_ID'], df['Season_ID']

        X_train, y_train, c_train, p_train, s_train = split_xy(train_df)
        X_val, y_val, c_val, p_val, s_val = split_xy(val_df)
        X_test, y_test, c_test, p_test, s_test = split_xy(test_df)

        X_train = X_train.fillna(X_train.median())
        X_val   = X_val.fillna(X_train.median())
        X_test  = X_test.fillna(X_train.median())

        X_train = self.scaler.fit_transform(X_train)
        X_val = self.scaler.transform(X_val)
        X_test = self.scaler.transform(X_test)

        device = self.cfg.DEVICE

        return (
            torch.tensor(X_train, dtype=torch.float32).to(device),
            torch.tensor(y_train.values, dtype=torch.float32).to(device),
            torch.tensor(c_train.values, dtype=torch.long).to(device),
            torch.tensor(p_train.values, dtype=torch.long).to(device),
            torch.tensor(s_train.values, dtype=torch.long).to(device),

            torch.tensor(X_val, dtype=torch.float32).to(device),
            torch.tensor(y_val.values, dtype=torch.float32).to(device),
            torch.tensor(c_val.values, dtype=torch.long).to(device),
            torch.tensor(p_val.values, dtype=torch.long).to(device),
            torch.tensor(s_val.values, dtype=torch.long).to(device),

            torch.tensor(X_test, dtype=torch.float32).to(device),
            torch.tensor(y_test.values, dtype=torch.float32).to(device),
            torch.tensor(c_test.values, dtype=torch.long).to(device),
            torch.tensor(p_test.values, dtype=torch.long).to(device),
            torch.tensor(s_test.values, dtype=torch.long).to(device),
        )

The pipeline emphasizes **relative performance within season–conference groups** and **temporal consistency**, which aligns with how All-Star selection is actually determined. It intentionally separates representation (features + embeddings) from interaction (handled later by attention), keeping preprocessing simple while preserving the structure needed for group-based learning.

| Stage        | Component                  | Operation | Meaning |
|-------------|---------------------------|----------|--------|
| Load        | CSV → DataFrame           | read_csv | Loads raw season-level player data |
| Encoding    | Conf_ID, Pos_ID           | int cast | Converts categorical indicators to embeddings |
| Encoding    | Season_ID                 | year offset | Normalizes season index relative to first year |
| Feature     | Usage                     | FGA + 0.44·FTA | Proxy for offensive involvement |
| Feature     | Usage × Win               | Usage * Win% | Scales volume by team success |
| Feature     | Impact_on_winning         | PTS * Win% | Combines scoring with team performance |
| Normalize   | PTS_conf_z                | z-score (group) | Measures scoring relative to peers in same season/conf |
| Normalize   | PTS_conf_share            | share of max | Captures dominance within group |
| Normalize   | Impact_conf_z             | z-score (group) | Relative impact vs peers |
| Temporal    | AllStar_Last_Year         | shift(1) | Encodes prior selection |
| Temporal    | AllStar_last_2            | rolling sum | Captures short-term momentum |
| Ordering    | Player, Season            | sort | Ensures temporal features are valid |
| Split       | Train / Val / Test        | time-based | Prevents leakage across seasons |
| Clean       | Missing values            | median fill | Robust to outliers |
| Scale       | StandardScaler            | fit/transform | Normalizes feature magnitudes |
| Structure   | Drop columns              | remove IDs | Prevents leakage from identifiers |
| Tensorize   | NumPy → Torch             | to(device) | Moves data to GPU/CPU |
| Output      | (X, y, conf, pos, season) | tuple | Preserves structure for grouped modeling |

## Neural Network Model: 

In [56]:
class AllStarNN(nn.Module):
    """
        Neural network for structured All-Star selection.
        Architecture:
        - Tabular features + learned embeddings (conference, position, season)
        - MLP for feature transformation
        - Multi-head self-attention over players within a group
        - Two output heads: starters and reserves
        Design intent:
        - Embeddings encode discrete structure
        - MLP builds player-level representations
        - Attention models competition within a selection pool
        - Separate heads reflect different selection roles (starter vs reserve)
    """
    def __init__(self, input_dim, cfg):
        """
            Initializes model components.
            Args:
                input_dim (int): number of continuous input features
                cfg: configuration object
            Components:
                - Embeddings:
                    conf_emb   → conference (2 → 4)
                    pos_emb    → position (2 → 4)
                    season_emb → season index (≤50 → 8)
                - MLP:
                    Transforms concatenated features into a latent representation
                - Attention:
                    Models interactions between players within the same group
                - Output heads:
                    starter_head → scores for starter selection
                    reserve_head → scores for reserve selection
            Notes:
                - ATTN_DIM defines the shared interaction space
                - LayerNorm + residual connections stabilize attention
        """
        super().__init__()

        self.conf_emb = nn.Embedding(2, 4)
        self.pos_emb = nn.Embedding(2, 4)
        self.season_emb = nn.Embedding(50, 8)
        self.ln1 = nn.LayerNorm(cfg.ATTN_DIM)
        self.ln2 = nn.LayerNorm(cfg.ATTN_DIM)
        self.ff = nn.Sequential(
            nn.Linear(cfg.ATTN_DIM, cfg.ATTN_DIM),
            nn.ReLU(),
            nn.Linear(cfg.ATTN_DIM, cfg.ATTN_DIM)
        )

        prev = input_dim + 16

        layers = []
        for h in cfg.HIDDEN_DIMS:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(cfg.DROPOUT)]
            prev = h

        self.mlp = nn.Sequential(*layers)
        self.to_attn = nn.Linear(prev, cfg.ATTN_DIM)

        self.attn = nn.MultiheadAttention(
            embed_dim=cfg.ATTN_DIM,
            num_heads=cfg.ATTN_HEADS,
            batch_first=True
        )

        self.starter_head = nn.Linear(cfg.ATTN_DIM, 1)
        self.reserve_head = nn.Linear(cfg.ATTN_DIM, 1)

    def forward_group(self, x, conf, pos, season):
        """
            Forward pass over a single (season, conference) group.
                Args:
                    x      : player feature tensor [N, input_dim]
                    conf   : conference indices [N]
                    pos    : position indices [N]
                    season : season indices [N]
                Steps:
                    1. Move inputs to model device
                    2. Concatenate features with embeddings
                    3. Pass through MLP → latent representation
                    4. Project to attention space
                    5. Apply self-attention across players in the group
                    6. Apply feedforward + residual + normalization
                    7. Produce separate scores for starters and reserves
                Returns:
                    starter_scores [N, 1]
                    reserve_scores [N, 1]
            Key idea:
                Scores are not computed independently—each player’s score depends
                on the full group via attention, reflecting competitive selection.
        """
        device = next(self.parameters()).device  

        x = x.to(device)

        conf = conf.to(device=device, dtype=torch.long)
        pos = pos.to(device=device, dtype=torch.long)
        season = season.to(device=device, dtype=torch.long)

        h = torch.cat([
            x,
            self.conf_emb(conf),
            self.pos_emb(pos),
            self.season_emb(season)
        ], dim=1)

        h = self.mlp(h)
        h = self.to_attn(h)
        # Attention operates over players within a group, allowing the model
        # to learn relative importance and competition dynamics.

        h = h.unsqueeze(0)
        attn_out, _ = self.attn(h, h, h)
        h = self.ln1(h + attn_out)

        ff_out = self.ff(h)
        h = self.ln2(h + ff_out)

        h = h.squeeze(0)

        return self.starter_head(h), self.reserve_head(h)
        """
            The model separates representation (MLP) from interaction (attention),
            allowing it to first understand players individually and then evaluate
            them relative to their competition.
        """

The model first builds a per-player representation, then refines it through attention over the entire group. Final predictions are split into starter and reserve scores, reflecting the structure of the selection process. Attention is applied across players within the same season–conference group, allowing the model to learn relative importance rather than absolute thresholds.

| Stage        | Component            | Operation | Meaning |
|-------------|----------------------|----------|--------|
| Input       | Raw Features + IDs   | concat | Combines numeric stats with embeddings |
| Embedding   | Conf Embedding       | Emb(2 → 4) | Encodes conference context |
| Embedding   | Position Embedding   | Emb(2 → 4) | Encodes backcourt/frontcourt role |
| Embedding   | Season Embedding     | Emb(50 → 8) | Captures temporal variation |
| Backbone    | MLP                  | Linear → ReLU → Dropout | Learns nonlinear feature interactions |
| Projection  | To Attention Space   | Linear → ATTN_DIM | Maps features into shared interaction space |
| Interaction | Self-Attention       | MultiheadAttention | Models relationships between players in group |
| Residual    | Add + LayerNorm      | h + attn(h) | Stabilizes attention updates |
| Feedforward | FFN                  | Linear → ReLU → Linear | Refines representations post-attention |
| Residual    | Add + LayerNorm      | h + ff(h) | Further stabilizes representation |
| Output      | Starter Head         | Linear → 1 | Scores likelihood of starter selection |
| Output      | Reserve Head         | Linear → 1 | Scores likelihood of reserve selection |
| Structure   | Group Forward Pass   | batch = group | Processes players jointly, not independently |

## Loss Function Definitions: 

In [59]:
def soft_selection(scores, k, temp):
    """
        Differentiable approximation of top-k selection.
        Converts raw scores into a soft allocation over players using a
        temperature-scaled softmax, then scales the distribution to sum to k.
        Args:
            scores : [N] raw logits
            k      : number of selections (e.g., 2 starters, 7 reserves)
            temp   : temperature controlling sharpness
        Returns:
            z : [N] soft selection weights summing to k
        Intuition:
            - As temp ↓ → distribution becomes sharper (closer to hard top-k)
            - As temp ↑ → distribution becomes smoother
        This allows gradients to flow through a top-k-like operation.
    """
    scores = scores.view(-1)
    probs = torch.softmax(scores / temp, dim=0)
    return k * probs 

def selection_loss(scores, labels, k, temp):
    """
        Loss enforcing correct soft top-k selection.
        Compares the model’s soft selection (z) to a normalized target
        distribution over true All-Stars.
        Args:
            scores : [N] model outputs
            labels : [N] binary indicators (1 = All-Star)
            k      : number of selections
            temp   : softmax temperature
        Returns:
            scalar loss
        Mechanism:
            - z = soft_selection(scores)
            - target = evenly distributes k mass over true positives
            - loss = mean squared error between z and target
        Notes:
            - If no positives exist in a group, loss is zero
            - Encourages correct allocation of selection mass, not just ranking
        Key idea:
            The model is trained to "spend" its k selections on the correct players.
    """
    scores = scores.view(-1)
    labels = labels.view(-1)

    if labels.sum() == 0:
        return torch.zeros(1, device=scores.device)

    z = soft_selection(scores, k, temp)

    target = k * labels / (labels.sum() + 1e-8)

    return torch.mean((z - target) ** 2)

def pairwise_loss(scores, labels, margin=0.01):
    """
        Margin-based pairwise ranking loss.
        Encourages positive examples to score higher than negative ones
        by at least a fixed margin.
        Args:
            scores : [N] model outputs
            labels : [N] binary indicators
            margin : minimum desired separation
        Returns:
            scalar loss
        Mechanism:
            - For all (pos, neg) pairs:
                loss = max(0, margin - (score_pos - score_neg))
            - Penalizes cases where positives do not sufficiently outrank negatives
        Notes:
            - Returns zero if no valid pairs exist
            - Operates within each group independently
        Key idea:
            Enforces relative ordering, complementing selection_loss which
            enforces allocation.
    """
    scores = scores.view(-1)
    labels = labels.view(-1)

    pos = scores[labels == 1]
    neg = scores[labels == 0]

    if len(pos) == 0 or len(neg) == 0:
        return torch.zeros(1, device=scores.device)

    diff = pos.unsqueeze(1) - neg.unsqueeze(0)
    return torch.relu(margin - diff).mean()

    """
        These losses serve complementary roles:
        - selection_loss → enforces *who gets selected*
        - pairwise_loss  → enforces *who ranks above whom*

        Together, they align the model with both the combinatorial structure
        (top-k selection) and the ordinal structure (ranking) of the task.
    """

$$
\mathcal{L}
=\left(
\lambda_{\text{BCE}} \cdot \mathcal{L}_{\text{BCE}} \right)
+
\left(\lambda_{\text{pair}} \cdot \mathcal{L}_{\text{pair}}\right)
+
\mathcal{L}_{\text{top-k}}
$$

Rather than predicting independent probabilities, the model is trained to allocate a fixed selection budget across players and rank true All-Stars above others within each group.

The loss combines a soft top-k selection objective with a pairwise ranking constraint. The selection term ensures the correct number of players are chosen, while the pairwise term enforces relative ordering between All-Stars and non-All-Stars. 

| Component     | Function          | Operation | Meaning |
|--------------|------------------|----------|--------|
| Selection    | Soft Selection    | softmax(scores / temp) | Converts scores into a soft top-k distribution |
| Selection    | Scaling           | k · probs | Ensures total mass equals number of selections |
| Selection    | Target Distribution | k · labels / sum(labels) | Ground-truth allocation over true All-Stars |
| Selection    | Loss              | MSE(z, target) | Penalizes deviation from correct selection proportions |
| Ranking      | Pairwise Loss     | max(0, margin − (pos − neg)) | Enforces All-Star scores > non-All-Star scores |
| Ranking      | Margin            | 0.01 | Minimum separation between positive and negative scores |
| Stability    | Temperature       | score / temp | Controls sharpness of selection distribution |
| Edge Case    | Empty Group       | return 0 | Avoids invalid loss when no positives exist |

## Training Loop Definition: 

In [64]:
class Trainer:
    """
        Training loop for the neural network model.
        Combines multiple objectives to align training with the structured
        All-Star selection task:
        - soft top-k selection (who gets chosen)
        - pairwise ranking (who beats whom)
        - BCE calibration (probability quality)
        - structural constraints (position balance, role separation)
    """
    def __init__(self, cfg):
        """
            Initializes trainer with configuration and loss functions.
            Uses BCEWithLogitsLoss for stable probability calibration.
        """
        self.cfg = cfg
        self.bce = nn.BCEWithLogitsLoss()

    def train(self, model,
              X_train, y_train, c_train, p_train, s_train,
              X_val, y_val, c_val, p_val, s_val,
              train_groups, val_groups):
        """
            Trains the model over grouped data.
            Args:
                model        : neural network
                X_*, y_*     : features and labels
                c_*, p_*, s_ : categorical indices (conference, position, season)
                *_groups     : grouped indices (per season, conference)
            Training is performed per group to preserve selection structure.
        """
        device = self.cfg.DEVICE
        model = model.to(device)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=self.cfg.LR,
            weight_decay=self.cfg.WEIGHT_DECAY
        )

        for epoch in range(self.cfg.EPOCHS):
            model.train()
            total_loss = 0

            for idx in train_groups:
                Xg = X_train[idx]
                yg = y_train[idx]
                cg = c_train[idx]
                pg = p_train[idx]
                sg = s_train[idx]

                bc = (pg == 1)
                fc = (pg == 0)
                """
                    Players are split by position to enforce:
                    - 2 backcourt starters
                    - 3 frontcourt starters
                    This mirrors real roster constraints directly in the loss.
                """

                optimizer.zero_grad()

                s, r = model.forward_group(Xg, cg, pg, sg)
                """
                    Model produces two score vectors:
                        - s: starter scores
                        - r: reserve scores
                    These are optimized separately but combined later.
                """
                loss = torch.zeros(1, device=Xg.device)

                if bc.sum() > 0:
                    loss += selection_loss(
                        s[bc], yg[bc], k=2, temp=self.cfg.TOPK_TEMP
                    )
                    loss += self.cfg.PAIRWISE_WEIGHT * pairwise_loss(s[bc], yg[bc])

                if fc.sum() > 0:
                    loss += selection_loss(
                        s[fc], yg[fc], k=3, temp=self.cfg.TOPK_TEMP
                    )
                    loss += self.cfg.PAIRWISE_WEIGHT * pairwise_loss(s[fc], yg[fc])

                loss += selection_loss(
                    r, yg, k=7, temp=self.cfg.TOPK_TEMP
                )
                loss += self.cfg.PAIRWISE_WEIGHT * pairwise_loss(r, yg)
                """
                    Starter losses:
                        - Backcourt: k = 2
                        - Frontcourt: k = 3
                    Reserve loss:
                        - Full group: k = 7
                        Each uses:
                        - selection_loss → enforces allocation
                        - pairwise_loss  → enforces ordering
                """
                combined_scores = (s + r).squeeze()
                loss += self.cfg.BCE_WEIGHT * self.bce(combined_scores, yg)
                """
                    Combined score represents overall selection strength.
                    BCE loss ensures scores remain probabilistically meaningful,
                    not just relatively ordered.
                """
                z_total = soft_selection(
                    combined_scores,
                    k=12,
                    temp=self.cfg.TOPK_TEMP
                )

                target_bc = 2 / 12
                actual_bc = z_total[pg == 1].sum() / 12

                pos_penalty = (actual_bc - target_bc) ** 2

                loss += 0.005 * pos_penalty  
                z_starters = torch.zeros_like(s.squeeze())

                if bc.sum() > 0:
                    z_starters[bc] = soft_selection(
                        s[bc], k=2, temp=self.cfg.TOPK_TEMP
                    )

                if fc.sum() > 0:
                    z_starters[fc] = soft_selection(
                        s[fc], k=3, temp=self.cfg.TOPK_TEMP
                    )

                z_reserves = soft_selection(
                    r.squeeze(), k=7, temp=self.cfg.TOPK_TEMP
                )

                overlap = (z_starters * z_reserves).sum()
                loss += 0.005 * overlap  
                """
                    Prevents the model from assigning the same players to both
                    Starter and reserve roles.
                    Overlap penalty discourages redundant allocation across heads.
                """
                loss.backward()
                optimizer.step()
                """
                    Standard gradient-based optimization:
                        - AdamW for stability + regularization
                        - Gradients accumulate per group
                """
                total_loss += loss.item()

            model.eval()
            scores = torch.zeros_like(y_val)

            with torch.no_grad():
                for idx in val_groups:
                    s_, r_ = model.forward_group(
                        X_val[idx], c_val[idx], p_val[idx], s_val[idx]
                    )
                    scores[idx] = (0.3 * s_ + 0.7 * r_).squeeze()

            auc = roc_auc_score(y_val.cpu(), scores.cpu())

            print(f"Epoch {epoch+1:03d} | Loss {total_loss:.2f} | Val AUC {auc:.4f}")
            """
                Validation is performed using combined scores.
                AUC is used as a ranking metric to monitor generalization,
                independent of hard selection.
            """

        return model
        """
            The training objective is explicitly structured to match the task:
            - selection_loss → enforces top-k constraints
            - pairwise_loss  → enforces relative ranking
            - BCE            → maintains calibration
            - penalties      → enforce roster structure
            This avoids treating the problem as independent classification
            and instead models it as constrained, grouped selection.
        """

The objective is not purely predictive. It enforces combinatorial structure (top-k selection), relative ordering (pairwise ranking), and global constraints (roster composition) during training. Training operates at the group level, optimizing a structured objective that combines selection constraints, ranking consistency, and probabilistic calibration. The model is explicitly guided to produce valid All-Star rosters rather than independent predictions.

| Component        | Step                  | Operation | Meaning |
|------------------|----------------------|----------|--------|
| Optimization     | Optimizer            | AdamW | Adaptive optimization with weight decay for stability |
| Optimization     | Learning Rate        | cfg.LR | Controls update step size |
| Optimization     | Weight Decay         | cfg.WEIGHT_DECAY | Prevents overfitting via parameter shrinkage |
| Training Loop    | Group Iteration      | loop over (season, conference) | Ensures learning happens within selection pools |
| Forward Pass     | Dual Heads           | (s, r) = model(...) | Produces starter and reserve scores |
| Selection Loss   | Starters (BC/FC)     | top-k MSE | Enforces correct number of starters per position group |
| Selection Loss   | Reserves             | top-k MSE | Enforces correct number of reserves |
| Ranking Loss     | Pairwise             | hinge(pos − neg) | Forces All-Stars to rank above non-All-Stars |
| Calibration      | BCE Loss             | BCE(s + r, y) | Maintains probabilistic interpretability |
| Constraint       | Position Balance     | (actual − target)^2 | Encourages correct backcourt/frontcourt proportions |
| Constraint       | Overlap Penalty      | sum(z_starters · z_reserves) | Prevents selecting same player twice |
| Regularization   | Loss Weights         | weighted sum | Balances selection, ranking, and calibration |
| Backprop         | Gradient Step        | loss.backward() + step() | Updates model parameters |
| Validation       | Score Aggregation    | 0.3·s + 0.7·r | Combines starter/reserve signals |
| Evaluation       | Metric               | AUC | Measures ranking quality |

## Evaluation Function Definition: 

In [68]:
def evaluate(model, X_test, y_test, c_test, p_test, s_test, test_df, cfg):
    """
        Evaluates the trained model under structured selection constraints.
        This function performs:
            1. Group-wise inference (season, conference)
            2. Score aggregation (starter + reserve heads)
            3. Hard roster selection (top-12 with positional constraints)
            4. Metric computation at global and grouped levels
        Args:
            model   : trained neural network
            X_test  : feature tensor
            y_test  : ground truth labels
            c_test  : conference indices
            p_test  : position indices
            s_test  : season indices
            test_df : original dataframe (for grouping + metadata)
            cfg     : configuration object
    """
    device = cfg.DEVICE
    model.eval()

    test_df = test_df.reset_index(drop=True)
    scores = torch.zeros(len(test_df), device=device)

    with torch.no_grad():
        """
            Inference is performed per group to preserve competition structure.
            Scores are computed as a weighted combination:
                0.3 * starter_score + 0.7 * reserve_score
            This reflects that most selections are reserves (7 vs 5 starters),
            while still incorporating starter importance.
        """
        groups = build_group_indices(test_df, device)

        for idx in groups:
            """
                Applies hard roster constraints:
                Per (season, conference):
                - Select top 2 backcourt starters
                - Select top 3 frontcourt starters
                - Select top 7 remaining players as reserves
                This converts continuous scores into discrete roster decisions.
            """
            s_, r_ = model.forward_group(
                X_test[idx],
                c_test[idx],
                p_test[idx],
                s_test[idx]
            )
            scores[idx] = torch.sigmoid((0.3*s_ + 0.7*r_).squeeze())

    scores_np = scores.cpu().numpy()
    y_np = y_test.cpu().numpy()
    test_df['score'] = scores_np
    test_df['pred'] = 0

    for (season, conf), group in test_df.groupby(['Season Ending Year', 'Conference_East']):
        
        bc = group[group['PosGroup_Backcourt'] == 1]
        fc = group[group['PosGroup_Backcourt'] == 0]

        starters = pd.concat([
            bc.sort_values('score', ascending=False).head(2),
            fc.sort_values('score', ascending=False).head(3)
        ])

        remaining = group.drop(index=starters.index)
        reserves = remaining.sort_values('score', ascending=False).head(7)

        selected = pd.concat([starters, reserves])
        test_df.loc[selected.index, 'pred'] = 1

    y_pred_global = test_df['pred'].values
    """
        Global metrics evaluate overall performance:
            - AUC: ranking quality (independent of selection threshold)
            - Accuracy: correctness of final selections
            - Precision: fraction of selected players who are true All-Stars
            - Recall: fraction of true All-Stars selected
            - F1: balance between precision and recall
    """


    print("\n=== FINAL RESULTS ===")
    print("AUC:", roc_auc_score(y_np, scores_np))
    print("Accuracy:", accuracy_score(y_np, y_pred_global))
    print("Precision:", precision_score(y_np, y_pred_global, zero_division=0))
    print("Recall:", recall_score(y_np, y_pred_global, zero_division=0))
    print("F1:", f1_score(y_np, y_pred_global, zero_division=0))

    print("\n=== Per-Season (Conference Split) ===")

    season_conf_metrics = []

    for (season, conf), group in test_df.groupby(['Season Ending Year', 'Conference_East']):
        """
            Per (season, conference) metrics assess consistency across groups.
            Additional metrics:
                - TopK Recall: fraction of true All-Stars captured
                - Top12 Accuracy: quality of selected roster (precision at k=12)
            These better reflect the structured nature of the task than global metrics alone.
        """
        y_g = group['All Star'].values
        p_g = group['pred'].values
        score_g = group['score'].values

        auc = roc_auc_score(y_g, score_g)
        acc = accuracy_score(y_g, p_g)
        prec = precision_score(y_g, p_g, zero_division=0)
        rec = recall_score(y_g, p_g, zero_division=0)
        f1 = f1_score(y_g, p_g, zero_division=0)

        true_allstars = y_g.sum()
        selected_correct = ((p_g == 1) & (y_g == 1)).sum()
        topk_recall = selected_correct / (true_allstars + 1e-8)

        total_selected = p_g.sum()
        top12_acc = selected_correct / (total_selected + 1e-8)

        season_conf_metrics.append((auc, acc, prec, rec, f1, topk_recall, top12_acc))

        print(
            f"{season} Conf {conf} | "
            f"AUC {auc:.4f} | "
            f"Acc {acc:.3f} | "
            f"P {prec:.3f} | R {rec:.3f} | F1 {f1:.3f} | "
            f"TopK {topk_recall:.3f} | Top12Acc {top12_acc:.3f}"
        )

    print("\n=== Per-Season (Overall) ===")
    season_metrics = []
    for season, group in test_df.groupby('Season Ending Year'):
        """
            Per-season metrics aggregate performance across conferences.
            This highlights temporal stability and identifies year-specific variation.
        """
        y_g = group['All Star'].values
        p_g = group['pred'].values
        score_g = group['score'].values

        auc = roc_auc_score(y_g, score_g)
        acc = accuracy_score(y_g, p_g)
        prec = precision_score(y_g, p_g, zero_division=0)
        rec = recall_score(y_g, p_g, zero_division=0)
        f1 = f1_score(y_g, p_g, zero_division=0)

        true_allstars = y_g.sum()
        selected_correct = ((p_g == 1) & (y_g == 1)).sum()
        topk_recall = selected_correct / (true_allstars + 1e-8)

        total_selected = p_g.sum()
        top12_acc = selected_correct / (total_selected + 1e-8)

        season_metrics.append((auc, acc, prec, rec, f1, topk_recall, top12_acc))

        print(
            f"{season} | "
            f"AUC {auc:.4f} | "
            f"Acc {acc:.3f} | "
            f"P {prec:.3f} | R {rec:.3f} | F1 {f1:.3f} | "
            f"TopK {topk_recall:.3f} | Top12Acc {top12_acc:.3f}"
        )

    season_metrics = np.array(season_metrics)
    """
        Average metrics summarize performance across seasons.
        This provides a stable estimate of model behavior under
        realistic selection constraints.
    """
    print("\n=== Avg Per-Season Metrics ===")
    print("AUC        :", season_metrics[:, 0].mean())
    print("Accuracy   :", season_metrics[:, 1].mean())
    print("Precision  :", season_metrics[:, 2].mean())
    print("Recall     :", season_metrics[:, 3].mean())
    print("F1         :", season_metrics[:, 4].mean())
    print("TopK Recall:", season_metrics[:, 5].mean())
    print("Top12 Acc  :", season_metrics[:, 6].mean())

    """
        Evaluation mirrors the true task: ranking players is not sufficient—
        the model must also construct valid rosters under fixed constraints.
    """

The model is not evaluated as a standard classifier. Performance is measured after enforcing the discrete selection constraint, making the metrics reflect actual roster construction rather than independent predictions. Evaluation mirrors the training structure: predictions are made within each (season, conference) group and converted into a valid 12-player roster. Metrics are reported both globally and at the group level to capture ranking quality and selection accuracy.

| Component        | Step                  | Operation | Meaning |
|------------------|----------------------|----------|--------|
| Inference        | Group Forward Pass   | (s, r) = model(...) | Computes starter and reserve scores per group |
| Score Aggregation| Weighted Combine     | 0.3·s + 0.7·r | Prioritizes reserve signal while retaining starter signal |
| Calibration      | Sigmoid              | σ(score) | Converts logits into probabilities |
| Grouping         | Selection Pools      | (season, conference) | Ensures predictions are made within valid All-Star pools |
| Selection        | Starters             | top-2 BC, top-3 FC | Enforces positional constraints |
| Selection        | Reserves             | top-7 remaining | Completes 12-player roster |
| Prediction       | Binary Labels        | pred ∈ {0,1} | Marks selected players as All-Stars |
| Global Metric    | AUC                  | roc_auc | Measures ranking quality over all players |
| Global Metric    | Accuracy             | correct / total | Measures overall classification correctness |
| Global Metric    | Precision            | TP / (TP + FP) | Measures selection quality |
| Global Metric    | Recall               | TP / (TP + FN) | Measures coverage of true All-Stars |
| Global Metric    | F1 Score             | harmonic mean | Balances precision and recall |
| Group Metric     | Per (Season, Conf)   | metrics per group | Evaluates local selection performance |
| Group Metric     | Per Season           | metrics per season | Evaluates yearly consistency |
| Ranking Metric   | Top-K Recall         | correct / true | Fraction of true All-Stars selected |
| Selection Metric | Top-12 Accuracy      | correct / 12 | Precision within selected roster |
| Aggregation      | Mean Metrics         | average over seasons | Stabilizes evaluation across years |

## Runtime: 

In [69]:
"""
End-to-end training pipeline for the neural network model.
Workflow:
    1. Initialize configuration and fix randomness
    2. Load and preprocess data
    3. Perform time-based train / validation / test split
    4. Convert data into tensors and structured inputs
    5. Build group indices (season, conference) for grouped training
    6. Initialize model and apply Kaiming weight initialization
    7. Train model using structured losses and validation monitoring
Key design choices:
    - Grouped training:
        Each forward pass operates on a full (season, conference) pool,
        allowing the model to learn selection dynamics rather than
        independent predictions.
    - Feature decomposition:
        Inputs are split into:
            - continuous features (X)
            - categorical embeddings (conference, position, season)
    - Weight initialization:
        He initialization stabilizes training for deep ReLU networks.
    - Structured optimization:
        Training combines:
            - soft top-k selection losses (roster constraints)
            - pairwise ranking losses (relative ordering)
            - BCE loss (global calibration)
    - Validation:
        AUC is tracked per epoch to monitor ranking quality.
Outputs:
    - Trained neural network model
    - Epoch-wise loss and validation AUC trajectory
Notes:
    - Training is fully GPU-compatible if available
    - Group indices are essential: they define the competitive context
    - Model learns both "who is good" and "who beats whom" within a pool
"""
cfg = Config()
set_seed(cfg.SEED)

pipeline = DataPipeline(cfg)
df = pipeline.load(DATA_PATH)
train_df, val_df, test_df = pipeline.split(df)

data = pipeline.prepare(train_df, val_df, test_df)

(X_train, y_train, c_train, p_train, s_train,
 X_val, y_val, c_val, p_val, s_val,
 X_test, y_test, c_test, p_test, s_test) = data

train_groups = build_group_indices(train_df, cfg.DEVICE)
val_groups   = build_group_indices(val_df, cfg.DEVICE)

In [70]:
model = AllStarNN(X_train.shape[1], cfg)

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

model.apply(init_weights)

trainer = Trainer(cfg)

model = trainer.train(
    model,
    X_train, y_train, c_train, p_train, s_train,
    X_val, y_val, c_val, p_val, s_val,
    train_groups, val_groups
)

Epoch 001 | Loss 18.02 | Val AUC 0.9811
Epoch 002 | Loss 9.88 | Val AUC 0.9836
Epoch 003 | Loss 8.85 | Val AUC 0.9844
Epoch 004 | Loss 8.23 | Val AUC 0.9849
Epoch 005 | Loss 7.98 | Val AUC 0.9848
Epoch 006 | Loss 7.65 | Val AUC 0.9849
Epoch 007 | Loss 7.47 | Val AUC 0.9847
Epoch 008 | Loss 7.29 | Val AUC 0.9847
Epoch 009 | Loss 7.02 | Val AUC 0.9849
Epoch 010 | Loss 6.83 | Val AUC 0.9852
Epoch 011 | Loss 6.70 | Val AUC 0.9853
Epoch 012 | Loss 6.65 | Val AUC 0.9856
Epoch 013 | Loss 6.55 | Val AUC 0.9857
Epoch 014 | Loss 6.40 | Val AUC 0.9860
Epoch 015 | Loss 6.23 | Val AUC 0.9865
Epoch 016 | Loss 6.22 | Val AUC 0.9866
Epoch 017 | Loss 6.08 | Val AUC 0.9868
Epoch 018 | Loss 6.03 | Val AUC 0.9868
Epoch 019 | Loss 6.03 | Val AUC 0.9869
Epoch 020 | Loss 6.05 | Val AUC 0.9873
Epoch 021 | Loss 5.85 | Val AUC 0.9875
Epoch 022 | Loss 5.85 | Val AUC 0.9873
Epoch 023 | Loss 5.80 | Val AUC 0.9874
Epoch 024 | Loss 5.81 | Val AUC 0.9874
Epoch 025 | Loss 5.69 | Val AUC 0.9878
Epoch 026 | Loss 5.61 | 

In [71]:
evaluate(
    model,
    X_test,
    y_test,
    c_test,
    p_test,
    s_test,
    test_df,
    cfg
)


=== FINAL RESULTS ===
AUC: 0.9827137802118109
Accuracy: 0.9739938080495356
Precision: 0.8333333333333334
Recall: 0.7547169811320755
F1: 0.7920792079207921

=== Per-Season (Conference Split) ===
2022 Conf False | AUC 0.9771 | Acc 0.976 | P 0.833 | R 0.769 | F1 0.800 | TopK 0.769 | Top12Acc 0.833
2022 Conf True | AUC 0.9939 | Acc 0.990 | P 1.000 | R 0.857 | F1 0.923 | TopK 0.857 | Top12Acc 1.000
2023 Conf False | AUC 0.9856 | Acc 0.959 | P 0.750 | R 0.643 | F1 0.692 | TopK 0.643 | Top12Acc 0.750
2023 Conf True | AUC 0.9808 | Acc 0.956 | P 0.667 | R 0.615 | F1 0.640 | TopK 0.615 | Top12Acc 0.667
2024 Conf False | AUC 0.9899 | Acc 0.968 | P 0.750 | R 0.750 | F1 0.750 | TopK 0.750 | Top12Acc 0.750
2024 Conf True | AUC 0.9914 | Acc 0.981 | P 0.917 | R 0.786 | F1 0.846 | TopK 0.786 | Top12Acc 0.917
2025 Conf False | AUC 0.9873 | Acc 0.975 | P 0.833 | R 0.769 | F1 0.800 | TopK 0.769 | Top12Acc 0.833
2025 Conf True | AUC 0.9938 | Acc 0.986 | P 0.917 | R 0.846 | F1 0.880 | TopK 0.846 | Top12Acc

## Results: 

### Design Philosophy

The core idea behind this model is that All-Star selection is not an independent classification problem, but a structured selection problem. Players are not evaluated in isolation; they are competing within a fixed, constrained pool defined by season and conference.

Rather than predicting probabilities and thresholding, the model is designed to directly approximate the selection process. This is done by enforcing three properties during training: (1) the correct number of players must be selected (top-k structure), (2) true All-Stars should rank above non-All-Stars (pairwise ordering), and (3) outputs should remain probabilistically meaningful (calibration via BCE).

The architecture reflects this structure. A shared representation is learned for each player, but attention operates at the group level, allowing players to be evaluated relative to others in the same selection pool. Separate heads for starters and reserves introduce role-specific scoring, while still allowing interaction through shared features and attention.

Loss design is intentionally compositional. The selection loss enforces global constraints, the pairwise loss enforces local ordering, and BCE stabilizes optimization. Additional penalties ensure that roster construction remains valid (positional balance and no overlap between starter/reserve selections).

Overall, the model is built to align the training objective with the actual decision process. Instead of hoping a classifier generalizes to structured selection, the structure is explicitly encoded into both the architecture and the loss.

### Training Summary

| Metric        | Value |
|---------------|-------|
| Initial Loss  | 18.02 |
| Final Loss    | 4.63  |
| Best Val AUC  | 0.9888 |
| Epochs        | 80    |

### Final Model Performance

| Category        | Metric        | Value |
|-----------------|--------------|-------|
| Global          | AUC          | 0.9827 |
| Global          | Accuracy     | 0.9740 |
| Global          | Precision    | 0.8333 |
| Global          | Recall       | 0.7547 |
| Global          | F1 Score     | 0.7921 |
| Per-Season Avg  | AUC          | 0.9857 |
| Per-Season Avg  | Accuracy     | 0.9739 |
| Per-Season Avg  | Precision    | 0.8333 |
| Per-Season Avg  | Recall       | 0.7553 |
| Per-Season Avg  | F1 Score     | 0.7924 |
| Selection       | Top-K Recall | 0.7553 |
| Selection       | Top-12 Acc   | 0.8333 |

### Per-Season Performance

| Season | AUC    | Accuracy | Precision | Recall | F1   |
|--------|--------|----------|-----------|--------|------|
| 2022   | 0.9816 | 0.983    | 0.917     | 0.815  | 0.863 |
| 2023   | 0.9798 | 0.957    | 0.708     | 0.630  | 0.667 |
| 2024   | 0.9911 | 0.975    | 0.833     | 0.769  | 0.800 |
| 2025   | 0.9903 | 0.981    | 0.875     | 0.808  | 0.840 |



## Model Comparison: Neural Network vs Logistic Regression

The neural network extends the linear baseline by explicitly modeling the structure of the selection process rather than relying on feature engineering alone. While logistic regression achieves strong performance through carefully constructed, group-centered features, it remains limited to additive relationships and cannot adapt to context-dependent interactions between players.

The neural network addresses this by learning representations over grouped player sets. Embeddings encode categorical structure (conference, position, season), while attention allows each player’s evaluation to depend on the surrounding pool—effectively modeling competition within a selection group. The training objective further aligns with the task by combining soft top-k selection, pairwise ranking, and probabilistic calibration, directly optimizing for roster construction rather than independent classification.

Empirically, this results in improved recall and F1, particularly in edge cases where multiple candidates have similar profiles and selection depends on nuanced trade-offs. Conceptually, the key distinction is that logistic regression treats players independently and relies on engineered proxies for comparison, whereas the neural network learns these comparisons implicitly through interaction-aware representations.

As a result, the neural network is not simply a more complex model, but a better-aligned one: it reflects the grouped, competitive, and constrained nature of All-Star selection.